# GraphRAG con Neo4j + Ollama Embeddings

Demo de GraphRAG: indexa chunks de conocimiento como entidades y relaciones
en Neo4j, y realiza retrieval híbrido (vectorial + grafo).

In [ ]:
import os
import json
from neo4j import GraphDatabase
from langchain_ollama import OllamaEmbeddings
import numpy as np

In [ ]:
# Conexión a Neo4j
uri = os.getenv("NEO4J_URI", "bolt://localhost:7687")
user = os.getenv("NEO4J_USER", "neo4j")
password = os.getenv("NEO4J_PASSWORD", "neo4j")

driver = GraphDatabase.driver(uri, auth=(user, password))
driver.verify_connectivity()
print("Conectado a Neo4j")
driver.close()

In [ ]:
# Embeddings locales con Ollama
embeddings = OllamaEmbeddings(
    model=os.getenv("EMBEDDINGS_MODEL", "qwen3-embedding:latest"),
    base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
)
print("Embeddings cargados")

In [ ]:
# Cargar chunks existentes del disco
chunks_dir = "../data/chunks"
chunks = []
for fname in os.listdir(chunks_dir):
    fpath = os.path.join(chunks_dir, fname)
    if os.path.isfile(fpath):
        with open(fpath, "r", encoding="utf-8") as f:
            chunks.append({"file": fname, "content": f.read()})

print(f"Cargados {len(chunks)} chunks")

In [ ]:
# Indexar chunks como entidades en Neo4j con embeddings
def index_chunks_in_neo4j(chunks):
    with driver.session() as session:
        for chunk in chunks[:20]:
            emb = embeddings.embed_query(chunk["content"])
            session.run(
                """
                MERGE (c:Chunk {file: $file})
                SET c.content = $content,
                    c.embedding = $embedding
                """,
                file=chunk["file"],
                content=chunk["content"],
                embedding=emb,
            )
    print("Chunks indexados en Neo4j")

index_chunks_in_neo4j(chunks)

In [ ]:
# GraphRAG query: embedding + traversal
def graphrag_query(question: str, top_k: int = 3):
    q_emb = embeddings.embed_query(question)
    
    with driver.session() as session:
        # 1. Similaridad vectorial en Neo4j
        vector_results = session.run(
            """
            MATCH (c:Chunk)
            WITH c, vector.similarity.cosine(c.embedding, $embedding) AS score
            WHERE score > 0.5
            RETURN c.file AS file, c.content AS content, score
            ORDER BY score DESC
            LIMIT $top_k
            """,
            embedding=q_emb,
            top_k=top_k,
        )
        
        for record in vector_results:
            print(f"\n--- {record['file']} (score: {record['score']:.3f}) ---")
            print(record["content"][:300])

graphrag_query("¿Qué sabes sobre Python?")

In [ ]:
driver.close()
print("Conexión cerrada")